In [1]:
import numpy as np
import pandas as pd
import torch
import torch.backends.cuda
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score
import matplotlib.pyplot as plt
import re
# import warnings
# warnings.filterwarnings('ignore')

device = torch.device('cuda')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

Device: cuda
PyTorch: 2.12.1+rocm7.2


In [3]:
df = pd.read_csv('../datasets/heusler_magnetic_cleaned.csv')
df_pde = pd.read_csv('../datasets/PubChemElements_all.csv').fillna(0)

# Normalize continuous targets
eform_max = df['e_form'].abs().max()
latt_max = df['latt const'].abs().max()

def extract_element_features(formula, df_pde):
    matches = re.findall(r'([A-Z][a-z]*)(\d*)', formula)
    features = []
    props = ['AtomicNumber','Electronegativity','AtomicRadius','IonizationEnergy',
             'ElectronAffinity','MeltingPoint','BoilingPoint','Density']
    for element, count in matches:
        n = int(count) if count else 1
        row = df_pde[df_pde['Symbol'] == element]
        f = []
        for p in props:
            val = row[p].values[0]
            max_val = df_pde[p].abs().max()
            f.append(val/max_val if max_val != 0 else 0)
        f.append(row['GroupBlock'].values[0] == 'Transition Metal')
        for _ in range(n):
            features.append(f)
    return np.array(features)

idx_train_val, idx_test = train_test_split(np.arange(len(df)), test_size=0.1, random_state=42)
idx_train, idx_val = train_test_split(idx_train_val, test_size=0.111, random_state=42)

def get_full_transformer_tensors(df_subset):
    element_features = []
    for formula in df_subset['formula']:
        features = extract_element_features(formula, df_pde)
        element_features.append(features)

    padded_features = []
    masks = []
    for feats in element_features:
        pad_len = 4 - len(feats)
        mask = [1] * len(feats) + [0] * pad_len
        if pad_len > 0:
            feats = np.vstack([feats, np.zeros((pad_len, feats.shape[1]))])
        padded_features.append(feats)
        masks.append(mask)

    X_t = torch.tensor(np.array(padded_features), dtype=torch.float32, device=device)
    mask_t = torch.tensor(np.array(masks), dtype=torch.bool, device=device)

    heusler_map = {'Full Heusler': 0, 'Half Heusler': 1, 'Inverse Heusler': 2}
    htypes = [heusler_map.get(t, 0) for t in df_subset['heusler type']]
    htype_t = torch.tensor(htypes, dtype=torch.long, device=device)

    stabs = df_subset['stability'].astype(str).map({'TRUE': 1.0, 'FALSE': 0.0, 'True': 1.0, 'False': 0.0, '1': 1.0, '0': 0.0}).fillna(0.0).values
    stab_t = torch.tensor(stabs, dtype=torch.float32, device=device).unsqueeze(1)

    eforms = df_subset['e_form'].values / eform_max
    eform_t = torch.tensor(eforms, dtype=torch.float32, device=device).unsqueeze(1)

    lcs = df_subset['latt const'].values / latt_max
    lc_t = torch.tensor(lcs, dtype=torch.float32, device=device).unsqueeze(1)

    struct_map = {'D022': 0, 'L21': 1, 'C1b': 2, 'tetragonal': 3, 'Xa': 4}
    structs = [struct_map.get(s, 0) for s in df_subset['struct type']]
    struct_t = torch.tensor(structs, dtype=torch.long, device=device)

    return X_t, mask_t, htype_t, stab_t, eform_t, lc_t, struct_t

X_train_t, mask_train_t, htype_train_t, stab_train_t, ef_train_t, lc_train_t, struct_train_t = get_full_transformer_tensors(df.iloc[idx_train])
X_val_t, mask_val_t, htype_val_t, stab_val_t, ef_val_t, lc_val_t, struct_val_t = get_full_transformer_tensors(df.iloc[idx_val])
X_test_t, mask_test_t, htype_test_t, stab_test_t, ef_test_t, lc_test_t, struct_test_t = get_full_transformer_tensors(df.iloc[idx_test])

In [15]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=4):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))
        t = torch.arange(max_seq_len).type_as(inv_freq)
        freqs = torch.einsum('i,j->ij', t, inv_freq)
        self.register_buffer('cos', freqs.cos())
        self.register_buffer('sin', freqs.sin())

    def forward(self, x):
        return self.cos, self.sin

def apply_rotary_emb(x, cos, sin):
    d = x.shape[-1] // 2
    x1 = x[..., :d]
    x2 = x[..., d:]
    rotated = torch.cat([-x2, x1], dim=-1)
    cos = torch.cat([cos, cos], dim=-1).unsqueeze(0)
    sin = torch.cat([sin, sin], dim=-1).unsqueeze(0)
    return (x * cos) + (rotated * sin)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )

    def forward(self, x, mask, cos, sin):
        # Pre-LN
        h = self.norm1(x)
        # Apply RoPE to queries and keys
        q = apply_rotary_emb(h, cos, sin)
        k = apply_rotary_emb(h, cos, sin)
        attn_out, _ = self.attention(q, k, h)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x

class HeuslerTransformer(nn.Module):
    def __init__(self, input_dim=9, d_model=128, n_layers=4, n_heads=4):
        super().__init__()
        self.atom_proj = nn.Linear(input_dim, d_model)
        self.type_emb = nn.Embedding(3, d_model)
        self.rope = RotaryPositionalEmbedding(d_model)

        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads) for _ in range(n_layers)])
        self.norm_final = nn.LayerNorm(d_model)

        self.head_struct = nn.Linear(d_model, 5)
        self.head_stab = nn.Linear(d_model, 1)
        self.head_eform = nn.Linear(d_model, 1)
        self.head_lc = nn.Linear(d_model, 1)

    def forward(self, atoms, mask, htype):
        x = self.atom_proj(atoms)
        t_emb = self.type_emb(htype).unsqueeze(1)
        x = x + t_emb

        cos, sin = self.rope(x)
        for layer in self.layers:
            x = layer(x, mask, cos, sin)

        x = self.norm_final(x)
        mask_expanded = mask.unsqueeze(-1).float()
        pooled = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)

        return self.head_struct(pooled), self.head_stab(pooled), self.head_eform(pooled), self.head_lc(pooled)

    def multi_task_loss(self, p_struct, p_stab, p_ef, p_lc, t_struct, t_stab, t_ef, t_lc):
        l_str = F.cross_entropy(p_struct, t_struct)
        l_sta = F.binary_cross_entropy_with_logits(p_stab, t_stab)
        l_ef = F.mse_loss(p_ef, t_ef)
        l_lc = F.mse_loss(p_lc, t_lc)
        return l_str + l_sta + l_ef + l_lc, l_str.item(), l_sta.item(), l_ef.item(), l_lc.item()

model = HeuslerTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)

In [16]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 128
train_dataset = TensorDataset(X_train_t, mask_train_t, htype_train_t, struct_train_t, stab_train_t, ef_train_t, lc_train_t)
val_dataset = TensorDataset(X_val_t, mask_val_t, htype_val_t, struct_val_t, stab_val_t, ef_val_t, lc_val_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

EPOCHS = 300
history = {'loss':[], 'val_loss':[]}
best_val = float('inf')

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        b_X, b_mask, b_htype, b_struct, b_stab, b_ef, b_lc = batch

        optimizer.zero_grad()
        p_struct, p_stab, p_eform, p_lc = model(b_X, b_mask, b_htype)
        loss, _, _, _, _ = model.multi_task_loss(
            p_struct, p_stab, p_eform, p_lc,
            b_struct, b_stab, b_ef, b_lc
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()

    model.eval()
    val_loss_total = 0
    with torch.no_grad():
        for batch in val_loader:
            b_X, b_mask, b_htype, b_struct, b_stab, b_ef, b_lc = batch
            p_struct_v, p_stab_v, p_eform_v, p_lc_v = model(b_X, b_mask, b_htype)
            v_loss, _, _, _, _ = model.multi_task_loss(
                p_struct_v, p_stab_v, p_eform_v, p_lc_v,
                b_struct, b_stab, b_ef, b_lc
            )
            val_loss_total += v_loss.item()

    avg_train_loss = total_loss / len(train_loader)
    avg_val_loss = val_loss_total / len(val_loader)

    history['loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)

    if avg_val_loss < best_val:
        best_val = avg_val_loss
        torch.save(model.state_dict(), './transformer_best.pt')

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

model.load_state_dict(torch.load('./transformer_best.pt'))

Epoch 10/300 | Train Loss: 1.1454 | Val Loss: 1.0598
Epoch 20/300 | Train Loss: 1.1226 | Val Loss: 1.0271
Epoch 30/300 | Train Loss: 1.0904 | Val Loss: 1.0075
Epoch 40/300 | Train Loss: 1.0549 | Val Loss: 0.9721
Epoch 50/300 | Train Loss: 1.0440 | Val Loss: 0.9704
Epoch 60/300 | Train Loss: 1.0528 | Val Loss: 0.9251
Epoch 70/300 | Train Loss: 0.9576 | Val Loss: 0.8473
Epoch 80/300 | Train Loss: 0.9316 | Val Loss: 0.7761
Epoch 90/300 | Train Loss: 0.9354 | Val Loss: 0.7399
Epoch 100/300 | Train Loss: 0.8783 | Val Loss: 0.7503
Epoch 110/300 | Train Loss: 0.8016 | Val Loss: 0.6524
Epoch 120/300 | Train Loss: 0.7680 | Val Loss: 0.6386
Epoch 130/300 | Train Loss: 0.7476 | Val Loss: 0.6148
Epoch 140/300 | Train Loss: 0.7363 | Val Loss: 0.6098
Epoch 150/300 | Train Loss: 0.7245 | Val Loss: 0.5995
Epoch 160/300 | Train Loss: 0.7663 | Val Loss: 0.6755
Epoch 170/300 | Train Loss: 0.7664 | Val Loss: 0.5832
Epoch 180/300 | Train Loss: 0.6862 | Val Loss: 0.5740
Epoch 190/300 | Train Loss: 0.6189 | 

<All keys matched successfully>

In [17]:
test_dataset = TensorDataset(X_test_t, mask_test_t, htype_test_t, struct_test_t, stab_test_t, ef_test_t, lc_test_t)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
all_struct_pred, all_struct_true = [], []
all_stab_pred, all_stab_true = [], []
all_eform_pred, all_eform_true = [], []
all_lc_pred, all_lc_true = [], []

with torch.no_grad():
    for batch in test_loader:
        b_X, b_mask, b_htype, b_struct, b_stab, b_ef, b_lc = batch
        p_struct, p_stab, p_eform, p_lc = model(b_X, b_mask, b_htype)

        all_struct_pred.extend(torch.argmax(p_struct, dim=1).cpu().numpy().flatten())
        all_struct_true.extend(b_struct.cpu().numpy().flatten())

        all_stab_pred.extend(torch.sigmoid(p_stab).cpu().numpy().flatten())
        all_stab_true.extend(b_stab.cpu().numpy().flatten())

        all_eform_pred.extend(p_eform.cpu().numpy().flatten() * eform_max)
        all_eform_true.extend(b_ef.cpu().numpy().flatten() * eform_max)

        all_lc_pred.extend(p_lc.cpu().numpy().flatten() * latt_max)
        all_lc_true.extend(b_lc.cpu().numpy().flatten() * latt_max)

all_struct_pred = np.array(all_struct_pred)
all_struct_true = np.array(all_struct_true)
all_stab_pred = np.array(all_stab_pred)
all_stab_true = np.array(all_stab_true)
all_eform_pred = np.array(all_eform_pred)
all_eform_true = np.array(all_eform_true)
all_lc_pred = np.array(all_lc_pred)
all_lc_true = np.array(all_lc_true)

print(f"Structure Acc: {accuracy_score(all_struct_true, all_struct_pred):.4f}")
print(f"Stability Acc: {accuracy_score(all_stab_true, all_stab_pred>0.5):.4f}")
print(f"Latt Const R2: {r2_score(all_lc_true, all_lc_pred):.4f}")
print(f"E_Form R2:     {r2_score(all_eform_true, all_eform_pred):.4f}")

Structure Acc: 0.8624
Stability Acc: 0.8991
Latt Const R2: 0.0853
E_Form R2:     0.8805
